# 기억하는 튜터 · checkpointer 와 thread

강의 오후 ② 앞부분 (Academy 모듈 1 · L7 Agent Memory)

| 칸 | 보여 주는 것 |
| --- | --- |
| ① 기억 없이 | invoke 두 번은 서로 다른 실행이다. "방금 찾은 단어" 를 모른다 |
| ② checkpointer | `compile({ checkpointer })` 하고 같은 `thread_id` 로 부르면 앞의 State 에서 이어 간다 |
| ③ thread | 다른 `thread_id` 는 다른 대화다 |
| ④ checkpoint | 단계마다 State 가 저장돼 있다 |

오후 ① 에서 만든 튜터 Agent 를 그대로 가져왔습니다. 이번에는 두 번 이어서 물어보겠습니다. 모델은 실습과 같은 OpenRouter 의 `qwen/qwen3-32b` 이고, 접속 정보는 `lab/.env` 에서 읽습니다.

In [ ]:
// @@cell setup
import "npm:@langchain/core@1.2.14";
import { Annotation, StateGraph, START, END, MemorySaver } from "npm:@langchain/langgraph@1.4.18";

// 접속 정보: 실습과 같은 lab/.env 를 읽는다 (OpenRouter · qwen/qwen3-32b). 키는 화면에 찍지 않는다
const ENV: Record<string, string> = {};
try {
  for (const line of Deno.readTextFileSync("../lab/.env").split("\n")) {
    const m = line.match(/^\s*([A-Z_]+)\s*=\s*(.*)\s*$/);
    if (m) ENV[m[1]] = m[2];
  }
} catch { /* lab/.env 가 없으면 아래 기본값 */ }
const BASE_URL = ENV.LLM_BASE_URL ?? "https://openrouter.ai/api/v1";
const API_KEY  = ENV.LLM_API_KEY  ?? "";
const MODEL    = ENV.LLM_MODEL    ?? "qwen/qwen3-32b";
const OPTIONS  = { temperature: 0 };   // 시연이 매번 같게
console.log({ BASE_URL, MODEL, 키: API_KEY ? "있음" : "없음 — lab/.env 에 LLM_API_KEY 를 넣는다" });

// LangSmith (강사 시연용): demo/.env 에 키가 있으면 실행 기록(trace)이 LangSmith 에 남는다. 실습은 lab/.env 를 써서 남지 않는다
import { traceable } from "npm:langsmith@0.11.1/traceable";
try {
  for (const line of Deno.readTextFileSync(".env").split("\n")) {
    const m = line.match(/^\s*(LANGSMITH_[A-Z_]+)\s*=\s*(?:"([^"]*)"|([^#\s]*))/);   // "따옴표 값" 은 그대로, 아니면 # 주석 앞까지
    const value = m?.[2] ?? m?.[3];
    if (m && value) Deno.env.set(m[1], value);
  }
} catch { /* demo/.env 가 없으면 기록하지 않는다 */ }
if (!Deno.env.get("LANGSMITH_API_KEY")) Deno.env.set("LANGSMITH_TRACING", "false");   // 키가 없으면 끈다
// 모델을 부르는 함수를 감싸면 trace 에 "모델 호출" 한 칸으로 남는다. 기록을 안 켜면 그냥 원래 함수로 돈다
const asLLM = <F extends (...a: any[]) => Promise<any>>(fn: F, name: string) => traceable(fn, {
  name, run_type: "llm",
  processInputs: (i: any) => ({ messages: i.args ? i.args[0] : i.input }),
  processOutputs: (o: any) => ({ choices: [{ message: typeof o.outputs === "string" ? { role: "assistant", content: o.outputs } : o }] }),
});
console.log({ LangSmith: Deno.env.get("LANGSMITH_TRACING") === "true" && Deno.env.get("LANGSMITH_API_KEY") ? `기록함 · project ${Deno.env.get("LANGSMITH_PROJECT") ?? "default"}` : "기록 안 함 (demo/.env)" });

// @@cell chat
// 오전에 만든 chat() 그대로 — 다시 보낼 칸(role · content · tool_calls)만 돌려준다
async function callModel(messages: any[], tools?: any[]) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
    body: JSON.stringify({ model: MODEL, messages, tools, ...OPTIONS }),
  });
  const { role, content, tool_calls } = (await res.json()).choices[0].message;
  return { role, content: content?.trim() ?? null, tool_calls };
}
const chat = asLLM(callModel, "chat");   // LangSmith 에 모델 호출로 남긴다

// 도구 하나: 단어 뜻과 쉬운 말 찾기 (작은 사전이라 시연이 매번 같다)
const DICT: Record<string, { 뜻: string; 쉬운말: string }> = {
  ubiquitous: { 뜻: "어디에나 있는", 쉬운말: "everywhere" },
  mitigate:   { 뜻: "완화하다, 줄이다", 쉬운말: "reduce" },
  meticulous: { 뜻: "꼼꼼한", 쉬운말: "careful" },
  detrimental:{ 뜻: "해로운", 쉬운말: "harmful" },
};
const tools = [{
  type: "function",
  function: {
    name: "lookup_word",
    description: "영어 단어 하나의 뜻과, 바꿔 쓸 쉬운 말을 사전에서 찾는다",
    parameters: { type: "object", properties: { word: { type: "string", description: "찾을 영어 단어 하나" } }, required: ["word"] },
  },
}];
function lookupWord(word: string) {
  const w = word.toLowerCase().trim();
  return DICT[w] ? { word: w, ...DICT[w] } : { word: w, error: "사전에 없는 단어" };
}

const SYSTEM = { role: "system", content: "너는 IELTS 쓰기를 돕는 영어 튜터다. 단어 뜻은 추측하지 말고 lookup_word 로 확인한다. 한국어로 짧게 답한다." };
const ask = (q: string) => ({ messages: [SYSTEM, { role: "user", content: q }] });
function show(out: any) {
  const last = out.messages.at(-1);
  console.log(`messages ${out.messages.length}개 · 마지막: ${last.role}`);
  console.log(last.tool_calls?.length ? `  도구 요청 ${JSON.stringify(last.tool_calls.map((c: any) => c.function))}` : `  ${last.content}`);
}

// 그래프 그림: 노드·엣지 이름을 mermaid.ink 로 보내 PNG 로 받아 셀 출력에 띄운다. 받지 못하면 mermaid 글자로 찍는다
async function showGraph(graph: any, options: { xray?: boolean } = {}) {
  const drawable = await graph.getGraphAsync(options);   // xray: true 면 subgraph 안의 노드까지 그린다
  try {
    const png = await drawable.drawMermaidPng();
    await Deno.jupyter.display(Deno.jupyter.image(new Uint8Array(await png.arrayBuffer())));
  } catch {
    console.log(drawable.drawMermaid());
  }
}

## 튜터 Agent

오후 ① 의 ③ Agent 와 같습니다. tutor 가 모델을 부르고, 도구를 원하면 tools 로, 결과를 들고 다시 tutor 로 돌아옵니다. 이번에는 `compile()` 하기 전의 `builder` 까지만 만들어 두겠습니다. 같은 그래프를 기억 없이 한 번, 기억을 달고 한 번 compile 하려고요.

In [ ]:
// 오후 ① 의 튜터 Agent 그대로: tutor ⇢ tools → tutor
const State = Annotation.Root({
  messages: Annotation<any[]>({ reducer: (prev, next) => [...prev, ...next], default: () => [] }),
});
async function tutor(state: typeof State.State) {
  return { messages: [await chat(state.messages, tools)] };
}
function toolsNode(state: typeof State.State) {
  const calls = state.messages.at(-1).tool_calls;
  console.log(`▶ tools   ${calls.map((c: any) => c.function.arguments).join(" · ")}`);
  return { messages: calls.map((c: any) => ({
    role: "tool", tool_call_id: c.id, content: JSON.stringify(lookupWord(JSON.parse(c.function.arguments).word)),
  })) };
}
function route(state: typeof State.State) {
  return state.messages.at(-1).tool_calls?.length ? "tools" : END;
}
const builder = new StateGraph(State)
  .addNode("tutor", tutor)
  .addNode("tools", toolsNode)
  .addEdge(START, "tutor")
  .addConditionalEdges("tutor", route, ["tools", END])
  .addEdge("tools", "tutor");

## ① 기억 없이

먼저 ubiquitous 의 뜻을 찾아 달라고 하고, 이어서 "방금 찾은 단어로 예문을 만들어 달라" 고 해 보겠습니다.

In [ ]:
// 기억 없이: invoke 두 번은 서로 다른 실행이다
const agent = builder.compile();
await showGraph(agent);
show(await agent.invoke(ask("ubiquitous 뜻을 사전에서 찾아 줘")));
show(await agent.invoke(ask("방금 찾은 단어로 IELTS 에 쓸 만한 영어 예문 하나 만들어 줘")));

**돌려 보면** 첫 번째는 사전을 찾아서 잘 답합니다. 그런데 두 번째는 엉뚱한 단어로 예문을 만듭니다. resilience 같은, 처음 보는 단어를 사전에서 찾기도 합니다.

왜 그럴까요? invoke 한 번이 그래프 실행 한 번이고, **State 는 그 실행 안에서만 살아 있습니다.** 두 번째 invoke 는 완전히 새 State 로 시작하는, 서로 독립된 실행입니다. 첫 번째 대화를 전혀 모릅니다.

## ② checkpointer 와 thread_id

이걸 푸는 게 **checkpointer** 입니다. checkpointer 는 그래프가 단계를 하나 지날 때마다 그때의 State 를 저장합니다. 가장 쉬운 건 `MemorySaver`, 메모리에 저장하는 checkpointer 입니다.

달 때는 `compile({ checkpointer: new MemorySaver() })` 한 줄이면 됩니다. 그리고 부를 때 `thread_id` 를 같이 줍니다. thread 는 checkpoint 들을 묶는 대화 번호라고 보면 됩니다.

두 번째 질문은 새 user 메시지 하나만 넣습니다. 같은 thread 의 State 에 이어 붙습니다.

In [ ]:
// checkpointer 를 달고 compile 한다. 같은 thread_id 로 부르면 앞의 State 에서 이어 간다
const memoryAgent = builder.compile({ checkpointer: new MemorySaver() });
const thread1 = { configurable: { thread_id: "1" } };
show(await memoryAgent.invoke(ask("ubiquitous 뜻을 사전에서 찾아 줘"), thread1));
show(await memoryAgent.invoke({ messages: [{ role: "user", content: "방금 찾은 단어로 IELTS 에 쓸 만한 영어 예문 하나 만들어 줘" }] }, thread1));

**돌려 보면** 이번에는 두 번째 답이 ubiquitous 로 예문을 만듭니다. messages 도 5개에서 7개로 이어졌습니다. 앞의 State 를 들고 이어 갔기 때문입니다.

## ③ 다른 thread 는 다른 대화

같은 그래프, 같은 checkpointer 라도 `thread_id` 가 다르면 다른 대화입니다.

In [ ]:
// 다른 thread_id 는 다른 대화다
const thread2 = { configurable: { thread_id: "2" } };
show(await memoryAgent.invoke(ask("방금 찾은 단어로 IELTS 에 쓸 만한 영어 예문 하나 만들어 줘"), thread2));

**돌려 보면** thread 2 는 앞 대화를 모릅니다. 다시 엉뚱한 단어로 예문을 만듭니다. 학생마다, 대화마다 thread_id 를 따로 주면 됩니다.

## ④ 단계마다 저장된 checkpoint

checkpointer 가 정말 단계마다 저장했는지 보겠습니다. `getState` 는 thread 의 지금 State 를, `getStateHistory` 는 그동안 저장된 checkpoint 들을 최근 것부터 돌려줍니다.

In [ ]:
// checkpointer 는 단계마다 State 를 저장해 둔다. thread 1 에 쌓인 checkpoint 를 세어 보면
const snapshot = await memoryAgent.getState(thread1);
console.log(`지금 State: messages ${snapshot.values.messages.length}개 · 다음 노드: ${JSON.stringify(snapshot.next)}`);
let n = 0;
for await (const c of memoryAgent.getStateHistory(thread1)) {
  if (n < 6) console.log(`  checkpoint · messages ${c.values.messages?.length ?? 0}개 · 다음 ${JSON.stringify(c.next)}`);
  n++;
}
console.log(`thread 1 의 checkpoint ${n}개`);

**돌려 보면** checkpoint 마다 그 시점의 messages 수와 다음에 갈 노드가 남아 있습니다. 두 번 invoke 했는데 checkpoint 는 여러 개입니다. 노드를 하나 지날 때마다 하나씩 쌓였습니다.

이 checkpoint 가 있어서 "이어 가기" 가 됩니다. 뒤에서 그래프를 중간에 멈췄다가 이어 가는 것도 이 위에서 돌아갑니다.